# Notebook 7: Explainable AI (XAI) con LIME

## Corso: Big Data Analytics
## Università degli studi di Milano - Bicocca
## Corso di Laurea: Economia, Analisi dei Dati e Management

---

### Introduzione

Nei notebook precedenti abbiamo costruito modelli di Machine Learning per classificare testi (es. la sentiment analysis con BoW e TF-IDF nel Notebook 3). I modelli funzionano e ci danno predizioni, ma...

**...come facciamo a capire PERCHÉ un modello prende una certa decisione?**

In questo notebook esploreremo la **Explainable AI (XAI)** e in particolare uno degli strumenti più famosi e utili: **LIME** (Local Interpretable Model-agnostic Explanations).


# IMMAGINATE: il vostro modello classifica come SPAM una mail importante del vostro capo. COME FATE A CAPIRE COSA NON VA? QUALI PAROLE HANNO FUORVIATO IL MODELLO?

---


## 1.1 Cos'è l'Explainable AI (XAI)?

L'**Explainable AI** è il ramo dell'AI che si occupa di **rendere comprensibili** le decisioni dei modelli di Machine Learning agli esseri umani.

### Perché serve?

I modelli moderni (Deep Learning, Random Forest, Gradient Boosting, BERT, GPT...) sono **black box**: producono predizioni accurate, ma sono molto difficili da interpretare.

| Tipo di Modello | Esempi | Interpretabilità | Performance |
| :--- | :--- | :---: | :---: |
| **White Box** | Regressione Lineare, Decision Tree | ✅ Alta | ❌ Limitata |
| **Black Box** | Random Forest, Reti Neurali, BERT, GPT | ❌ Bassa | ✅ Alta |

### Il dilemma trasparenza vs accuratezza

Spesso esiste un **trade-off**: più il modello è performante, più è difficile capire perché fa una certa predizione.

L'XAI cerca di **rompere questo trade-off**: ottenere modelli accurati MA al tempo stesso interpretabili (almeno *a posteriori*).


## 1.2 Quando ci servono spiegazioni?

L'XAI non è un capriccio accademico. È una necessità pratica in molti contesti reali:

- **🏥 Sanità**: se un modello predice che un paziente ha una certa patologia, il medico vuole sapere PERCHÉ prima di prescrivere una terapia.
- **🏦 Finanza**: se la banca rifiuta un mutuo, deve spiegare il motivo per legge (GDPR, regolamenti antidiscriminazione).
- **⚖️ Giustizia**: i sistemi di valutazione del rischio di recidiva DEVONO essere trasparenti.
- **🐛 Debugging**: se il modello sbaglia, vogliamo capire perché per migliorarlo.
- **🎯 Bias detection**: scoprire se il modello ha imparato pregiudizi nascosti nei dati.

> **Esempio reale**: nel 2018 Amazon ha dovuto abbandonare il suo sistema di selezione automatica dei CV perché aveva imparato a discriminare le donne. Senza tecniche di XAI, questo bias sarebbe rimasto invisibile!

# QUALI ALTRI ESEMPI VI VENGONO IN MENTE IN CUI UN MODELLO ACCURATO MA NON SPIEGABILE NON BASTA?


## 2.1 LIME: Local Interpretable Model-agnostic Explanations

**LIME** è uno degli algoritmi di XAI più utilizzati. È stato pubblicato nel 2016 da Ribeiro, Singh e Guestrin nel paper *"Why Should I Trust You?"* e risolve un problema apparentemente impossibile:

> Come spiego le predizioni di un modello complesso (una scatola nera) usando un modello semplice (interpretabile)?

### Decifriamo il nome (acronimo molto denso!)

| Termine | Significato |
| :--- | :--- |
| **L**ocal | Spiega UNA singola predizione, non l'intero modello globalmente |
| **I**nterpretable | Usa un modello semplice (es. regressione lineare sparse) come surrogato |
| **M**odel-agnostic | Funziona con qualsiasi modello (BERT, RandomForest, anche GPT...) |
| **E**xplanations | Produce spiegazioni leggibili (parole evidenziate, feature con pesi) |


## 2.2 L'intuizione: l'approssimazione locale

L'idea geniale di LIME è semplicissima:

> Un modello complesso può essere **localmente** approssimato da un modello semplice.

### Una metafora

Immaginate la superficie di decisione di un modello come una catena montuosa: a livello globale è curva, frastagliata, irregolare. Ma se zoommate molto su un singolo punto, in un piccolo intorno il terreno sembra **piatto** (lineare).

Allo stesso modo:
- A livello globale, una Random Forest è imprevedibile.
- A livello locale (intorno a una specifica predizione) può essere ben approssimata da una regressione lineare.

LIME impara questa **approssimazione lineare locale** per spiegare la singola predizione, perdendo fedeltà globale ma guadagnando interpretabilità.


## 2.3 Le formule (senza paura!)

L'obiettivo di LIME è trovare una spiegazione $g$ che soddisfi due requisiti contrapposti:
1. Sia **fedele** al modello complesso $f$ nell'intorno locale dell'istanza $x$.
2. Sia **semplice** (e quindi interpretabile per un umano).

### La formula generale

$$\xi(x) \;=\; \arg\min_{g \in G} \; \mathcal{L}\bigl(f, g, \pi_x\bigr) \;+\; \Omega(g)$$

Decifriamola un pezzo alla volta:

- $f$: il modello black box che vogliamo spiegare (es. la nostra Random Forest, BERT, ecc.)
- $x$: l'istanza specifica per cui vogliamo la spiegazione (es. una singola recensione)
- $G$: la famiglia di modelli interpretabili (in pratica regressioni lineari sparse)
- $g$: il modello interpretabile specifico che andremo a scegliere (la spiegazione)
- $\pi_x$: una misura di **prossimità** che dice "quanto un punto perturbato è vicino a $x$"
- $\mathcal{L}(f, g, \pi_x)$: la **fedeltà locale** -- quanto bene $g$ approssima $f$ vicino a $x$
- $\Omega(g)$: la **complessità** di $g$ (es. numero di feature usate -- più $g$ è semplice, meglio è)

### In parole povere

> "Trova il modello semplice $g$ che approssima MEGLIO il modello complesso $f$ nelle vicinanze del punto $x$, mantenendolo il più semplice possibile."

### La fedeltà locale (più nel dettaglio)

$$\mathcal{L}(f, g, \pi_x) \;=\; \sum_{z \in Z} \pi_x(z) \cdot \bigl( f(z) - g(z') \bigr)^2$$

dove:
- $z$ sono **istanze perturbate** intorno a $x$ (testi con parole rimosse, immagini con patch nascoste, ecc.)
- $z'$ è la rappresentazione interpretabile di $z$ (es. presenza/assenza di parole, codificate come $0$/$1$)
- $\pi_x(z) = \exp\!\left(-\dfrac{D(x,z)^2}{\sigma^2}\right)$ è un **kernel gaussiano**: i punti vicini all'istanza originale pesano di più nella loss

In sostanza: stiamo facendo una regressione lineare pesata dove i pesi danno più importanza alle perturbazioni vicine all'input originale.


## 2.4 L'algoritmo passo per passo

Per spiegare una singola predizione, LIME esegue questi passaggi:

1. **Perturbazione**: genera molte versioni perturbate dell'input $x$.
   - **Per il testo**: rimuove casualmente alcune parole (la rappresentazione interpretabile è un vettore binario "parola presente/assente").
   - **Per immagini**: nasconde alcune patch (super-pixel) dell'immagine.
   - **Per dati tabellari**: campiona valori da una distribuzione vicina ai valori originali.

2. **Predizione**: dà ogni versione perturbata in pasto al modello black box $f$ e raccoglie le probabilità predette.

3. **Pesatura**: assegna più peso alle perturbazioni vicine all'input originale (kernel gaussiano $\pi_x$).

4. **Modello surrogato**: addestra un modello semplice (regressione lineare con regolarizzazione Lasso, che spinge molti coefficienti a zero) sui dati perturbati pesati.

5. **Spiegazione**: i coefficienti del modello lineare sono la spiegazione -- mostrano quali feature hanno spinto la predizione **verso** o **contro** la classe predetta.

> **Notate bene**: la spiegazione è **locale**. Le stesse feature potrebbero contare in modo diverso per un'altra istanza! Questo è sia un punto di forza (spiegazioni personalizzate) sia un limite (non ho una visione globale).


# 🔬 Parte pratica: spiegare un classificatore di Sentiment Analysis

Riprendiamo il classificatore di **sentiment analysis** che abbiamo costruito nel Notebook 3 (TF-IDF + Logistic Regression sulle `movie_reviews` di NLTK) e lo rendiamo "interpretabile" usando LIME.

### Il piano:
1. **Setup**: installiamo LIME e carichiamo il dataset
2. **Training**: addestriamo un classificatore di recensioni
3. **Spiegazione**: usiamo LIME per spiegare predizioni specifiche
4. **Insights**: estraiamo insight pratici:
   - Quali parole guidano le predizioni positive/negative?
   - Il modello ha imparato bias o shortcut?
   - Perché certi esempi sono classificati male?
   - Come reagisce a frasi "trabocchetto" (negazioni, sarcasmo)?


In [ ]:
# ==========================================
# 1. INSTALLAZIONE DI LIME
# ==========================================

# LIME non è incluso di default in Colab/Anaconda
# Lo installiamo con pip
# Il "!" è un comando per dire a Colab "esegui questo nel terminale"
!pip install lime --quiet

print("LIME installato correttamente!")


In [ ]:
# ==========================================
# 2. IMPORT E CARICAMENTO DEL DATASET
# ==========================================

import numpy as np
import pandas as pd
import random
import nltk
import matplotlib.pyplot as plt

# scikit-learn: il classificatore e gli strumenti di vettorizzazione
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, classification_report

# LIME: la nostra novità!
# LimeTextExplainer è la classe specifica per spiegare modelli di testo
from lime.lime_text import LimeTextExplainer

# NLTK: per scaricare il dataset di recensioni di film
nltk.download('movie_reviews', quiet=True)
nltk.download('stopwords', quiet=True)
from nltk.corpus import movie_reviews

# Settiamo i seed per riproducibilità (lo stesso esperimento darà sempre lo stesso risultato)
np.random.seed(42)
random.seed(42)

# Costruiamo il DataFrame con tutte le recensioni e i loro sentiment
# (è esattamente lo stesso codice che abbiamo usato nel Notebook 3)
documents = []
for category in movie_reviews.categories():
    for fileid in movie_reviews.fileids(category):
        documents.append({
            'text': movie_reviews.raw(fileid),     # Testo completo della recensione
            'sentiment': category                  # Etichetta: 'pos' o 'neg'
        })

df = pd.DataFrame(documents)
print(f"Dataset caricato: {df.shape[0]} recensioni")
print(f"Distribuzione classi:")
print(df['sentiment'].value_counts())
df.head(3)


In [ ]:
# ==========================================
# 3. SPLIT TRAIN/TEST
# ==========================================

# Split classico 80/20
# stratify=df['sentiment'] mantiene la stessa proporzione di pos/neg in train e test
# random_state=42 garantisce che lo split sia sempre lo stesso
X_train, X_test, y_train, y_test = train_test_split(
    df['text'],
    df['sentiment'],
    test_size=0.2,
    random_state=42,
    stratify=df['sentiment']
)

print(f"Train set: {len(X_train)} recensioni")
print(f"Test set:  {len(X_test)} recensioni")


In [ ]:
# ==========================================
# 4. PIPELINE: TF-IDF + LOGISTIC REGRESSION
# ==========================================

# Costruiamo una "pipeline" scikit-learn che concatena due step:
#   1. TfidfVectorizer: trasforma il testo in vettori TF-IDF
#   2. LogisticRegression: il classificatore vero e proprio
#
# PERCHÉ una pipeline e non i due step separati?
# Perché LIME ha bisogno di una funzione che prenda IN INPUT IL TESTO GREZZO
# e restituisca le probabilità di classificazione.
# La pipeline fa esattamente questo: testo grezzo -> vettore TF-IDF -> probabilità.
# Se passassimo direttamente il modello, dovremmo prima vettorizzare manualmente
# i testi che LIME genera durante le perturbazioni.

pipeline = make_pipeline(
    TfidfVectorizer(
        max_features=20000,    # Limitiamo il vocabolario alle 20k parole più frequenti
        stop_words='english',  # Rimuoviamo parole comuni ('the', 'is', ecc.)
        ngram_range=(1, 2)     # Usiamo unigrammi E bigrammi (es. 'not bad' come token unico)
    ),
    LogisticRegression(max_iter=1000, random_state=42)
)

# Addestriamo l'intera pipeline sui dati di training
print("Addestramento in corso... (qualche secondo)")
pipeline.fit(X_train, y_train)
print("Addestramento completato!")


In [ ]:
# ==========================================
# 5. VALUTAZIONE DEL MODELLO
# ==========================================

# Vediamo come si comporta sul test set
y_pred = pipeline.predict(X_test)

acc = accuracy_score(y_test, y_pred)
print(f"Accuratezza sul test set: {acc:.4f}\n")
print("Report dettagliato (precision, recall, f1):")
print(classification_report(y_test, y_pred))


## 4.1 Costruiamo il LimeTextExplainer

Ora arriva il bello! Usiamo LIME per **spiegare** le predizioni del nostro modello.

L'oggetto chiave è `LimeTextExplainer` (per il testo). Per altri tipi di dati ci sono:
- `LimeTabularExplainer` per dati tabellari (es. dataset Iris, Titanic, ecc.)
- `LimeImageExplainer` per immagini

I principi sono gli stessi, cambia solo il modo di perturbare l'input.


In [ ]:
# ==========================================
# 6. INIZIALIZZAZIONE DELL'EXPLAINER
# ==========================================

# Creiamo l'oggetto LimeTextExplainer
# class_names: i nomi delle classi nell'ORDINE che il modello usa
# (importante: se sbagliamo l'ordine, le spiegazioni saranno invertite!)

# Verifichiamo l'ordine delle classi imparato dal modello
print(f"Ordine classi del modello: {pipeline.classes_}")

# Usiamo lo stesso ordine
# bow=True dice a LIME di ignorare l'ordine delle parole (coerente con TF-IDF)
explainer = LimeTextExplainer(
    class_names=pipeline.classes_,
    bow=True
)

print("\nExplainer pronto!")


In [ ]:
# ==========================================
# 7. SELEZIONE DI UNA RECENSIONE POSITIVA
# ==========================================

# Convertiamo le Series in liste per accedere agli indici comodamente
test_reviews = X_test.tolist()
test_labels  = y_test.tolist()
test_preds   = y_pred.tolist()

# Cerchiamo l'indice di una recensione positiva ben classificata
idx_positiva = next(
    i for i, (true, pred) in enumerate(zip(test_labels, test_preds))
    if true == 'pos' and pred == 'pos'
)

review_pos = test_reviews[idx_positiva]
proba_pos  = pipeline.predict_proba([review_pos])[0]

print(f"=== RECENSIONE POSITIVA #{idx_positiva} ===")
print(f"Classe vera: pos | Predetta: pos")
print(f"Probabilità predette: neg={proba_pos[0]:.4f}, pos={proba_pos[1]:.4f}")
print(f"\nPrimi 500 caratteri della recensione:\n{review_pos[:500]}...")


In [ ]:
# ==========================================
# 8. GENERIAMO LA SPIEGAZIONE CON LIME
# ==========================================

# explain_instance è il cuore di LIME:
# - text_instance: la recensione da spiegare
# - classifier_fn: la funzione che fa le predizioni (deve restituire probabilità)
# - num_features: quante feature (parole) mostrare nella spiegazione
# - num_samples: quante perturbazioni generare (più sono, più stabile è la spiegazione,
#                ma più tempo ci mette)
#
# ⏳ ATTENZIONE: questa cella può richiedere 30-60 secondi
# perché LIME genera migliaia di varianti perturbate del testo
# e per ognuna fa una predizione con il nostro modello!

explanation_pos = explainer.explain_instance(
    text_instance=review_pos,
    classifier_fn=pipeline.predict_proba,
    num_features=10,       # Top-10 parole nella spiegazione
    num_samples=1500       # Numero di perturbazioni
)

print("Spiegazione generata!")


In [ ]:
# ==========================================
# 9. VISUALIZZIAMO LA SPIEGAZIONE
# ==========================================

# .show_in_notebook() è il modo più ricco di visualizzare la spiegazione:
# - A SINISTRA: probabilità predette dal modello
# - AL CENTRO: bar plot delle parole con i loro pesi
#              (verde/blu = a favore della classe positiva,
#               rosso/arancione = contro)
# - A DESTRA: il testo originale con highlighting colorato sulle parole influenti

explanation_pos.show_in_notebook(text=True)


In [ ]:
# ==========================================
# 10. ISPEZIONE NUMERICA DELLA SPIEGAZIONE
# ==========================================

# Possiamo anche estrarre i pesi numericamente
# .as_list() ritorna una lista di tuple (parola, peso)
# - Peso > 0: la parola spinge verso la classe POSITIVA
# - Peso < 0: la parola spinge verso la classe NEGATIVA

print("=== TOP 10 PAROLE PIÙ INFLUENTI ===\n")
print(f"{'PAROLA':<25}{'PESO':>10}    DIREZIONE")
print("-" * 55)
for parola, peso in explanation_pos.as_list():
    direzione = "→ POS" if peso > 0 else "→ NEG"
    print(f"{parola:<25}{peso:>+10.4f}    {direzione}")


### 💡 Cosa abbiamo imparato?

Guardando le parole evidenziate possiamo trarre alcuni **insight**:

- Le parole con **peso positivo** (verde/blu) spingono il modello a predire `pos`
- Le parole con **peso negativo** (rosso/arancione) spingono verso `neg`
- Se il modello si basa su parole semanticamente sensate (`great`, `wonderful`, `excellent`...) è un buon segno!
- Se invece troviamo parole "neutrali" tra le top (es. nomi propri, articoli, numeri) è un campanello d'allarme: il modello potrebbe essersi attaccato a feature spurie del dataset (data leakage, shortcut learning).

# DOMANDA: secondo voi quali parole DOVREBBERO comparire come "positive" in una recensione di film? E quali "negative"?


In [ ]:
# ==========================================
# 11. SPIEGHIAMO ANCHE UNA RECENSIONE NEGATIVA
# ==========================================

# Ripetiamo lo stesso processo per una recensione classificata correttamente come NEGATIVA
idx_negativa = next(
    i for i, (true, pred) in enumerate(zip(test_labels, test_preds))
    if true == 'neg' and pred == 'neg'
)

review_neg = test_reviews[idx_negativa]
proba_neg  = pipeline.predict_proba([review_neg])[0]

print(f"=== RECENSIONE NEGATIVA #{idx_negativa} ===")
print(f"Classe vera: neg | Predetta: neg")
print(f"Probabilità: neg={proba_neg[0]:.4f}, pos={proba_neg[1]:.4f}")
print(f"\nPrimi 400 caratteri:\n{review_neg[:400]}...\n")

# Generiamo e mostriamo la spiegazione
explanation_neg = explainer.explain_instance(
    text_instance=review_neg,
    classifier_fn=pipeline.predict_proba,
    num_features=10,
    num_samples=1500
)

explanation_neg.show_in_notebook(text=True)


## 5. L'insight più potente: capire PERCHÉ il modello SBAGLIA

LIME diventa veramente potente quando lo usiamo per analizzare gli **errori del modello**.

Cerchiamo recensioni che sono state classificate **erroneamente** e vediamo quali parole hanno fuorviato il modello. Questo è uno degli usi più pratici di LIME nella vita di un data scientist:

> "Il mio modello sbaglia su questi 50 esempi. Posso individuare un pattern comune negli errori? Se sì, posso correggerlo."


In [ ]:
# ==========================================
# 12. TROVIAMO UN ESEMPIO MAL CLASSIFICATO
# ==========================================

# Cerchiamo una recensione veramente positiva che il modello ha classificato come negativa
# (un "falso negativo": la recensione è positiva ma il modello la vede come negativa)

idx_errore = next(
    i for i, (true, pred) in enumerate(zip(test_labels, test_preds))
    if true == 'pos' and pred == 'neg'
)

review_errore = test_reviews[idx_errore]
proba_errore  = pipeline.predict_proba([review_errore])[0]

print(f"=== ESEMPIO MAL CLASSIFICATO #{idx_errore} ===")
print(f"Classe vera: pos | Predetta: neg  ❌")
print(f"Probabilità: neg={proba_errore[0]:.4f}, pos={proba_errore[1]:.4f}")
print(f"\nPrimi 600 caratteri:\n{review_errore[:600]}...\n")

# Spieghiamo questa predizione errata
explanation_errore = explainer.explain_instance(
    text_instance=review_errore,
    classifier_fn=pipeline.predict_proba,
    num_features=12,
    num_samples=1500
)

explanation_errore.show_in_notebook(text=True)


### 🐛 Cosa è andato storto?

Guardando la spiegazione possiamo provare a capire **perché** il modello ha sbagliato. Le ipotesi tipiche sono:

- La recensione è **effettivamente ambigua**: contiene critiche specifiche pur essendo nel complesso positiva.
- Il modello si è fissato su parole semanticamente "negative" usate però in **contesto positivo** (negazione, sarcasmo, "non è male", "didn't disappoint").
- Il modello ha imparato **shortcut su feature spurie**: nomi di registi/attori associati a film negativi nel training set, parole che capitano di più nelle recensioni negative ma sono semanticamente neutre, ecc.
- Il **TF-IDF con BoW non cattura il contesto**: per il modello "good" e "not good" sono entrambi solo "good".

Questo tipo di analisi è preziosissimo per:
1. **🔧 Debugging del modello** -- capire e correggere errori sistematici.
2. **🎯 Detection di bias** -- scoprire associazioni indesiderate.
3. **⚠️ Gestione del rischio** -- in applicazioni ad alto impatto (sanità, giustizia, finanza) sapere QUANDO non fidarsi della predizione.


In [ ]:
# ==========================================
# 13. PROBING: TESTIAMO IL MODELLO CON ESEMPI CUSTOM
# ==========================================

# Possiamo usare il modello (e poi LIME) per "interrogarlo" con frasi a piacere.
# Questo è utilissimo per testare ipotesi specifiche sul comportamento del modello.

frasi_test = [
    "this movie is absolutely amazing, brilliant and wonderful!",       # palesemente positiva
    "this movie is terrible, boring and awful.",                        # palesemente negativa
    "the movie was not bad at all, actually quite enjoyable.",          # NEGAZIONE: è positiva!
    "the movie was great. just kidding, it was awful.",                 # SARCASMO: è negativa
]

print("=== PREDIZIONI DEL MODELLO ===\n")
for frase in frasi_test:
    proba = pipeline.predict_proba([frase])[0]
    pred  = pipeline.classes_[np.argmax(proba)]
    print(f"Frase:        '{frase}'")
    print(f"Predizione:   {pred}  (neg={proba[0]:.3f}, pos={proba[1]:.3f})")
    print()


In [ ]:
# ==========================================
# 14. SPIEGAZIONE DI UNA FRASE CON NEGAZIONE
# ==========================================

# Vediamo cosa succede con la frase tricky:
#   "the movie was not bad at all, actually quite enjoyable."
# Questa frase è SEMANTICAMENTE POSITIVA, ma contiene la parola 'bad'.
# Il modello capirà la negazione di "bad"?

frase_tricky = "the movie was not bad at all, actually quite enjoyable."

# Con bigrammi (ngram_range=(1,2)) il modello PUÒ vedere "not bad" come token unico,
# quindi in linea teorica dovrebbe gestirlo. Vediamo se è così!

explanation_tricky = explainer.explain_instance(
    text_instance=frase_tricky,
    classifier_fn=pipeline.predict_proba,
    num_features=8,
    num_samples=2000
)

print("Spiegazione per:")
print(f"   '{frase_tricky}'\n")
explanation_tricky.show_in_notebook(text=True)


### 🤔 Riflessione: i limiti del Bag-of-Words

Senza usare bigrammi, in una frase come `"the movie was not bad at all"` un classificatore basato su BoW/TF-IDF unigram-only:

- vedrebbe `bad` come token isolato con peso negativo (preso da solo è negativo);
- non capirebbe che la `not` immediatamente precedente lo nega.

Con `ngram_range=(1, 2)` il modello PUÒ trattare `"not bad"` come bigramma unico, ma:
- deve aver visto abbastanza esempi di `"not bad"` nel training set per imparare che è positivo;
- non gestisce ancora costruzioni più complesse come `"the movie was great. just kidding, it was awful."` (sarcasmo).

> **Soluzione**: per gestire negazione complessa e sarcasmo serve passare a modelli **contestuali** (BERT, RoBERTa, modelli LLM) che vedremo nei prossimi notebook.

LIME è prezioso proprio perché ci mostra in modo **lampante** dove il modello si rompe -- e quindi quando serve cambiare approccio.


## 6. Pro e contro di LIME

### ✅ Vantaggi

- **Model-agnostic**: funziona con qualunque modello (anche GPT-4, BERT, modelli proprietari di cui non vediamo l'interno...).
- **Interpretazioni intuitive**: parole evidenziate, facili da spiegare anche a stakeholder non tecnici.
- **Local**: utile per giustificare singole decisioni a clienti/utenti finali.
- **Versatile**: testo, immagini, dati tabellari -- stesso framework concettuale.

### ⚠️ Limiti

- **Stabilità**: ripetendo lo stesso `explain_instance` si possono ottenere risultati LEGGERMENTE diversi (le perturbazioni sono casuali). Per spiegazioni stabili servono molti `num_samples`.
- **Solo locale**: una spiegazione vale per UNA istanza, non per il modello globale.
- **Costo computazionale**: ogni spiegazione richiede migliaia di chiamate al modello -- pesante per modelli lenti come BERT.
- **Dipendenza dalle perturbazioni**: la qualità dipende dal numero e dal tipo di perturbazioni.
- **Approssimazione lineare**: se l'intorno locale è davvero non-lineare, l'approssimazione perde di significato.

### Alternative

Esistono altre tecniche di XAI che vale la pena conoscere:

| Tecnica | Idea principale | Quando usarla |
| :--- | :--- | :--- |
| **SHAP** | Valori di Shapley dalla teoria dei giochi | Quando serve coerenza globale e garanzie matematiche |
| **Anchors** | Regole "if-then" sufficienti per la predizione | Quando vuoi regole esplicite da mostrare a un utente |
| **Integrated Gradients** | Gradient-based, per reti neurali | Quando hai accesso ai gradienti del modello |
| **Attention** | Pesi di attenzione (in Transformer) | Per modelli come BERT/GPT |
| **Counterfactuals** | "Cosa cambierebbe se modificassi X?" | Per scenari "what-if" |


## 🎯 Esercizi finali

Per consolidare la comprensione, provate a:

1. **Variate `num_samples`** (es. 100, 500, 5000) e osservate quanto cambia la spiegazione. Cosa succede al variare del numero di perturbazioni?

2. **Spiegate 5 recensioni diverse** del test set e confrontate le top-3 parole più influenti. Vi sembrano sensate? Trovate parole sospette?

3. **Costruite una frase a tradimento**: scrivete una recensione palesemente positiva ma usando parole tipicamente negative (`"not bad"`, `"didn't disappoint"`, `"hardly boring"`). Cosa predice il modello? Cosa dice LIME?

4. **Sostituite la `LogisticRegression` con un `RandomForestClassifier`**. LIME funziona ancora (sì, dovrebbe!). Le spiegazioni sono diverse? Più o meno stabili?

5. **Confronto con SHAP**: installate la libreria `shap` (`!pip install shap`) e provate a spiegare la stessa istanza con `shap.LinearExplainer`. Le spiegazioni concordano?

6. **Bias hunting**: cercate sistematicamente parole "neutrali" (nomi propri, anni, numeri) tra le top-feature di 20 spiegazioni casuali. Se il modello si basa frequentemente su queste, ha probabilmente imparato shortcut.

> 💡 **Nei prossimi notebook** estenderemo questi concetti a modelli più complessi (BERT/Transformer) e vedremo tecniche di XAI globali.

---

### Riferimenti

- Ribeiro, Singh, Guestrin (2016). *"Why Should I Trust You?": Explaining the Predictions of Any Classifier*. KDD 2016.
- Repository ufficiale di LIME: <https://github.com/marcotcr/lime>
- Documentazione: <https://lime-ml.readthedocs.io/>
